In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from sklearn.metrics import r2_score

import os
import sys

sys.path.append('/Users/kjesta/Desktop/Master prosjekt/Master_code/Lab/')

from funcs import *
from clean_and_combine import *

import matplotlib.style as mplstyle
mplstyle.use(["ggplot", "fast"])

import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

# Scaling waves

I have gathered the results of the waves tank experiments conducted by Maxime Leclerc during the spring semester 2025. My findings are in the folders /20_cm, /30_cm, /40_cm, where I assessed the waves ran for the three different water depths and for the 6 different frequencies per water depth. For each water depth and frequency configuration, Maxime ran the waves with and without plates mounted at the bottom, for 5 different positions of the probe array. Each configuration (water depth, plates/no plates, frequency, probe position) was run 3 times. So, in order to get an overview and to assess which runs gave meaningful damping, I picked out four probes per water depth - frequency - plate configuration (1 reference probe before the plates, and 3 probes evenly spaced over the plates). From this, I found that there were no damping which were within the accuracy of the probes for the deeper waters levels. 

However, I found that three of the waves ran on 20 cm gave damping. I will therefore scale the waves ran by Maxime on 20 cm to 30 cm. This leaves me with a set of experiments to run, where for the 30 cm we run the same wave as for 20 cm but scaled. That means that the viscous boundary layers relatively speaking becomes smaller, and we expect less damping. I will also try to increase the wave amplitude by 10 percent to ensure we are within the accuracy of the probes. This is a tradeoff, as then kH might become bigger.

## Methods used for the results:
For a given experiment of X cm water depth for a given frequency and amplitude I, 

1) Opened all three runs for experiments with both plates and empty tank.
2) Combined the three respective runs for experiment with plates and empty tank.
3) Subtracted the mean of the first 200 rows in the data to take out probe noise.
4) Applied a low-pass filter to smooth out outliers.
5) Took the mean of the three runs. So I was left with one dataset with plates and one without.
6) Selected 4 probes from the virtual array of 20 probes to look at. The placements of the probes chosen was then $11.075, 12.595, 13.81, 15.33$. The plates were placed between $12.4$ m and $15.45$ m.
7) Plotted the signal from the first probe, as this is not disturbed by the plates. From this plot, I visually decided at which point in time the wave train sabilized.
8) Used the timestamp of stable waves and cut off the first part of the signal for all probes. 
9) Tried to find a method for cutting off parts of the signal which contains reflected waves. With no luck. Perhaps because there were no waves that were stabilized + not reflected (due to the waves being long and the probes being placed in the middle of the tank).
10) Subtracted the mean of each probe signal from the data to again avoid gauge noise. Then found the standard deviation of each probe, and from there found the amplitude.
11) Saw how the amplitude changed from probe to probe.
12) Estimated wave number and frequency by Fourier transform. 
13) Used this to calculate the dimensionless variables, $kH$ and $ak$.

* Amplitude difference from before plates to the end of the plates: $ A_{diff} = A_{x=11.075 m} - A_{x=15.33 m}$
* Percentage damping due to plates = $\frac{A_{diff}^{plates + walls} - A_{diff}^{walls}}{A_{diff}^{plates + walls}}$

## Method for scaling

Take one original wave at depth $H_1$, with wave steepness $k_1*a_1$ and relative depth $k_1*H_1$. We want to scale this wave by keeping these two constant.
So for a new wave with depth $H_2$, we should have:

$k_1*a_1 = k_2*a_2$ and $k_1*H_1 = k_2*H_2$.

So,
$$k_2 = \frac{k_1 H_1}{H_2}$$
$$a_2 = \frac{k_1 a_1}{k_2} = \frac{a_1 H_2}{H_1}$$

And thus for the dispersion relation,
$$\omega_1^2 = gk_1 tanh(k_1 H_1)$$
$$\omega_2^2 = gk_2 tanh(k_2 H_2)$$

but with $k_1*H_1 = k_2*H_2$ we have that $tanh(k_1 H_1) = tanh(k_2 H_2)$.

So,
$$\omega_2^2 = g k_2 \frac{\omega_1^2}{g k_1} = \frac{k_2}{k_1}\omega_1^2$$
$$\omega_2 = \omega_1 \sqrt{\frac{k_2}{k_1}} = \omega_1 \sqrt{\frac{H_1}{H_2}}$$

### Reminder of wave properties
* The wavelength is $\lambda = \frac{2 \pi}{k}$
* Wave period is $T = \frac{2 \pi}{\omega} = \frac{1}{f}$
* Frequency is $f = \frac{1}{T}$
* Angular frequency $\omega = 2\pi f$
* Phase speed, $c = \frac{\omega}{k} = f \lambda$

## Scaled waves

In [7]:
original_frequencies = [1.53, 1.37, 1.22, 1.07, 0.92, 0.76]
original_amplitudes  = [0.009, 0.0083, 0.0074, 0.0064, 0.0053, 0.0043]
original_wavelengths = [0.63, 0.77, 0.91, 1.1, 1.33, 1.69]

original_wave_numbers = [2 * np.pi / l for l in original_wavelengths]

H_1 = 0.2  # original depth
H_2 = 0.3  
H_3 = 0.4  

results = []

for f, a, k, lamb in zip(original_frequencies, original_amplitudes, original_wave_numbers, original_wavelengths):
    w = 2 * np.pi * f

    # --- Scaling to 30 cm ---
    w_new_2 = w * np.sqrt(H_1 / H_2)
    f_new_2 = w_new_2 / (2 * np.pi)
    a_new_2 = a * H_2 / H_1
    k_new_2 = k * H_1 / H_2
    lamb_new_2 = 2 * np.pi / k_new_2
    kH_new_2 = k_new_2 * H_2
    ak_new_2 = a_new_2 * k_new_2

    # --- Scaling to 40 cm ---
    w_new_3 = w * np.sqrt(H_1 / H_3)
    f_new_3 = w_new_3 / (2 * np.pi)
    a_new_3 = a * H_3 / H_1
    k_new_3 = k * H_1 / H_3
    lamb_new_3 = 2 * np.pi / k_new_3
    kH_new_3 = k_new_3 * H_3
    ak_new_3 = a_new_3 * k_new_3

    results.append({
        'f 20 cm': f,
        'a 20 cm': a,
        'k 20 cm': k,
        'λ 20 cm': lamb,
        'f 30 cm': f_new_2,
        'a 30 cm': a_new_2,
        'k 30 cm': k_new_2,
        'λ 30 cm': lamb_new_2,
        'f 40 cm': f_new_3,
        'a 40 cm': a_new_3,
        'k 40 cm': k_new_3,
        'λ 40 cm': lamb_new_3,
        'kH': k * H_1,
        'ak': a * k,
    })

results_df_from_20cm = pd.DataFrame(results)

In [8]:
# Scaled results from 20 cm depth to 30 cm and 40 cm depths
results_df_from_20cm

,f 20 cm,a 20 cm,k 20 cm,λ 20 cm,f 30 cm,a 30 cm,k 30 cm,λ 30 cm,f 40 cm,a 40 cm,k 40 cm,λ 40 cm,kH,ak
0,1.53,0.0090,9.973310,0.63,1.249240,0.01350,6.648873,0.945,1.081873,0.0180,4.986655,1.26,1.994662,0.089760
1,1.37,0.0083,8.159981,0.77,1.118600,0.01245,5.439987,1.155,0.968736,0.0166,4.079990,1.54,1.631996,0.067728
2,1.22,0.0074,6.904599,0.91,0.996126,0.01110,4.603066,1.365,0.862670,0.0148,3.452300,1.82,1.380920,0.051094
3,1.07,0.0064,5.711987,1.10,0.873651,0.00960,3.807991,1.650,0.756604,0.0128,2.855993,2.20,1.142397,0.036557
4,0.92,0.0053,4.724199,1.33,0.751177,0.00795,3.149466,1.995,0.650538,0.0106,2.362100,2.66,0.944840,0.025038
5,0.76,0.0043,3.717861,1.69,0.620537,0.00645,2.478574,2.535,0.537401,0.0086,1.858931,3.38,0.743572,0.015987


In [27]:
# Modifying by increasing amplitude by 10%

proposed_frequencies = [1.5, 1.3, 1.2, 1.1, 0.9]
proposed_amplitudes  = [a * 1.2 for a in original_amplitudes[:5]] # increase by 20%
proposed_wavelengths = [0.63, 0.77, 0.91, 1.1, 1.33]
proposed_wave_numbers = [2 * np.pi / l for l in proposed_wavelengths]

H_1 = 0.2  # proposed depth
H_2 = 0.3  
H_3 = 0.4  

results = []

for f, a, k, lamb in zip(proposed_frequencies, proposed_amplitudes, proposed_wave_numbers, proposed_wavelengths):
    w = 2 * np.pi * f

    # --- Scaling to 30 cm ---
    w_new_2 = w * np.sqrt(H_1 / H_2)
    f_new_2 = w_new_2 / (2 * np.pi)
    a_new_2 = a * H_2 / H_1
    k_new_2 = k * H_1 / H_2
    lamb_new_2 = 2 * np.pi / k_new_2
    kH_new_2 = k_new_2 * H_2
    ak_new_2 = a_new_2 * k_new_2

    results.append({
        'f_20cm [s^1]': f,
        'a_20cm [m]': a,
        'k_20cm [m^-1]': k,
        'λ_20cm [m]': lamb,
        'f_30cm [s^1]': f_new_2,
        'a_30cm [m]': a_new_2,
        'k_30cm [m^-1]': k_new_2,
        'λ_30cm [m]': lamb_new_2,
        'kH': k * H_1,
        'ak': a * k,
    })

results_df_from_20cm_increased_amp = pd.DataFrame(results)

In [28]:
results_df_from_20cm_increased_amp

,f_20cm [s^1],a_20cm [m],k_20cm [m^-1],λ_20cm [m],f_30cm [s^1],a_30cm [m],k_30cm [m^-1],λ_30cm [m],kH,ak
0,1.5,0.01080,9.973310,0.63,1.224745,0.01620,6.648873,0.945,1.994662,0.107712
1,1.3,0.00996,8.159981,0.77,1.061446,0.01494,5.439987,1.155,1.631996,0.081273
2,1.2,0.00888,6.904599,0.91,0.979796,0.01332,4.603066,1.365,1.380920,0.061313
3,1.1,0.00768,5.711987,1.10,0.898146,0.01152,3.807991,1.650,1.142397,0.043868
4,0.9,0.00636,4.724199,1.33,0.734847,0.00954,3.149466,1.995,0.944840,0.030046


# ignore the below for now

### Results (derivations in notebooks: `\finding_waves\40_cm\40cm_fXX_AXX.ipynb`)

In [19]:
# 40 cm, frequency 1 Hz, Amplitude 0.3 V

H = 0.4
a = 0.018
f = 1
w = 2 * np.pi * f
k = 2*np.pi/1.46
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 1.000 Hz
Original wave number: 4.304 rad/m
Original wavelength: 1.460 m
Original amplitude: 0.018 m
Original kH: 1.721
Original ak: 0.077
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 1.155 Hz
New wave number: 5.738 rad/m
New wavelength: 1.095 m
New amplitude: 0.013 m
New kH: 1.721
New ak: 0.077
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 1.414 Hz
New wave number: 8.607 rad/m
New wavelength: 0.730 m
New amplitude: 0.009 m
New kH: 1.721
New ak: 0.077


In [20]:
# 40 cm, frequency 0.9 Hz, Amplitude 0.3 V

H = 0.4
a = 0.016
f = 0.9
w = 2 * np.pi * f
k = 2*np.pi/1.71
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.900 Hz
Original wave number: 3.674 rad/m
Original wavelength: 1.710 m
Original amplitude: 0.016 m
Original kH: 1.470
Original ak: 0.059
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 1.039 Hz
New wave number: 4.899 rad/m
New wavelength: 1.282 m
New amplitude: 0.012 m
New kH: 1.470
New ak: 0.059
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 1.273 Hz
New wave number: 7.349 rad/m
New wavelength: 0.855 m
New amplitude: 0.008 m
New kH: 1.470
New ak: 0.059


In [21]:
# 40 cm, frequency 0.8 Hz, Amplitude 0.3 V

H = 0.4
a = 0.014
f = 0.8
w = 2 * np.pi * f
k = 2*np.pi/2.03
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.800 Hz
Original wave number: 3.095 rad/m
Original wavelength: 2.030 m
Original amplitude: 0.014 m
Original kH: 1.238
Original ak: 0.043
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 0.924 Hz
New wave number: 4.127 rad/m
New wavelength: 1.522 m
New amplitude: 0.010 m
New kH: 1.238
New ak: 0.043
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 1.131 Hz
New wave number: 6.190 rad/m
New wavelength: 1.015 m
New amplitude: 0.007 m
New kH: 1.238
New ak: 0.043


In [22]:
# 40 cm, frequency 0.7 Hz, Amplitude 0.3 V

H = 0.4
a = 0.012
f = 0.7
w = 2 * np.pi * f
k = 2*np.pi/2.43
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.700 Hz
Original wave number: 2.586 rad/m
Original wavelength: 2.430 m
Original amplitude: 0.012 m
Original kH: 1.034
Original ak: 0.031
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 0.808 Hz
New wave number: 3.448 rad/m
New wavelength: 1.823 m
New amplitude: 0.009 m
New kH: 1.034
New ak: 0.031
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 0.990 Hz
New wave number: 5.171 rad/m
New wavelength: 1.215 m
New amplitude: 0.006 m
New kH: 1.034
New ak: 0.031


In [23]:
# 40 cm, frequency 0.6 Hz, Amplitude 0.3 V

H = 0.4
a = 0.01
f = 0.6
w = 2 * np.pi * f
k = 2*np.pi/2.95
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.600 Hz
Original wave number: 2.130 rad/m
Original wavelength: 2.950 m
Original amplitude: 0.010 m
Original kH: 0.852
Original ak: 0.021
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 0.693 Hz
New wave number: 2.840 rad/m
New wavelength: 2.212 m
New amplitude: 0.007 m
New kH: 0.852
New ak: 0.021
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 0.849 Hz
New wave number: 4.260 rad/m
New wavelength: 1.475 m
New amplitude: 0.005 m
New kH: 0.852
New ak: 0.021


In [ ]:
# 40 cm, frequency 0.5 Hz, Amplitude 0.3 V

H = 0.4
a = 0.0078
f = 0.5
w = 2 * np.pi * f
k = 2*np.pi/3.73
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 40 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 20 cm water depth---
H_new = 0.2
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 40 cm to 20 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.500 Hz
Original wave number: 1.685 rad/m
Original wavelength: 3.730 m
Original amplitude: 0.008 m
Original kH: 0.674
Original ak: 0.013
-----
Scaling results from 40 cm to 30 cm water depth:

New frequency: 0.577 Hz
New wave number: 2.246 rad/m
New wavelength: 2.797 m
New amplitude: 0.006 m
New kH: 0.674
New ak: 0.013
-----
Scaling results from 40 cm to 20 cm water depth:

New frequency: 0.707 Hz
New wave number: 3.369 rad/m
New wavelength: 1.865 m
New amplitude: 0.004 m
New kH: 0.674
New ak: 0.013


In [54]:
original_frequencies = [1.00, 0.90, 0.80, 0.70, 0.60, 0.50]
original_amplitudes  = [0.018, 0.016, 0.014, 0.012, 0.010, 0.0078]
original_wavelengths = [1.46, 1.71, 2.03, 2.43, 2.95, 3.73]

original_wave_numbers = [2 * np.pi / l for l in original_wavelengths]

H_1 = 0.4  # original depth
H_2 = 0.2  
H_3 = 0.3  

results = []

for f, a, k, lamb in zip(original_frequencies, original_amplitudes, original_wave_numbers, original_wavelengths):
    w = 2 * np.pi * f

    # --- Scaling to 20 cm ---
    w_new_2 = w * np.sqrt(H_1 / H_2)
    f_new_2 = w_new_2 / (2 * np.pi)
    a_new_2 = a * H_2 / H_1
    k_new_2 = k * H_1 / H_2
    lamb_new_2 = 2 * np.pi / k_new_2
    kH_new_2 = k_new_2 * H_2
    ak_new_2 = a_new_2 * k_new_2

    # --- Scaling to 30 cm ---
    w_new_3 = w * np.sqrt(H_1 / H_3)
    f_new_3 = w_new_3 / (2 * np.pi)
    a_new_3 = a * H_3 / H_1
    k_new_3 = k * H_1 / H_3
    lamb_new_3 = 2 * np.pi / k_new_3
    kH_new_3 = k_new_3 * H_3
    ak_new_3 = a_new_3 * k_new_3

    results.append({
        'f 40cm': f,
        'a 40cm': a,
        'k 40cm': k,
        'λ 40cm': lamb,
        'f 20cm': f_new_2,
        'a 20cm': a_new_2,
        'k 20cm': k_new_2,
        'λ 20cm': lamb_new_2,
        'f 30cm': f_new_3,
        'a 30cm': a_new_3,
        'k 30cm': k_new_3,
        'λ 30cm': lamb_new_3,
        'kH': k * H_1,
        'ak': a * k,
    })

results_df_from_40cm = pd.DataFrame(results)

### Results (derivations in notebooks: `\finding_waves\20_cm\20cm_fXX_AXX.ipynb`)

In [28]:
# 20 cm, frequency 0.76 Hz, Amplitude 0.15 V

H = 0.2
a = 0.0043
f = 0.76
w = 2 * np.pi * f
k = 2*np.pi/1.69
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 20 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 40 cm water depth---
H_new = 0.4
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 20 cm to 40 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.760 Hz
Original wave number: 3.718 rad/m
Original wavelength: 1.690 m
Original amplitude: 0.004 m
Original kH: 0.744
Original ak: 0.016
-----
Scaling results from 20 cm to 30 cm water depth:

New frequency: 0.621 Hz
New wave number: 2.479 rad/m
New wavelength: 2.535 m
New amplitude: 0.006 m
New kH: 0.744
New ak: 0.016
-----
Scaling results from 20 cm to 40 cm water depth:

New frequency: 0.537 Hz
New wave number: 1.859 rad/m
New wavelength: 3.380 m
New amplitude: 0.009 m
New kH: 0.744
New ak: 0.016


In [29]:
# 20 cm, frequency 0.92 Hz, Amplitude 0.15 V

H = 0.2
a = 0.0053
f = 0.92
w = 2 * np.pi * f
k = 2*np.pi/1.33
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 20 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 40 cm water depth---
H_new = 0.4
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 20 cm to 40 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 0.920 Hz
Original wave number: 4.724 rad/m
Original wavelength: 1.330 m
Original amplitude: 0.005 m
Original kH: 0.945
Original ak: 0.025
-----
Scaling results from 20 cm to 30 cm water depth:

New frequency: 0.751 Hz
New wave number: 3.149 rad/m
New wavelength: 1.995 m
New amplitude: 0.008 m
New kH: 0.945
New ak: 0.025
-----
Scaling results from 20 cm to 40 cm water depth:

New frequency: 0.651 Hz
New wave number: 2.362 rad/m
New wavelength: 2.660 m
New amplitude: 0.011 m
New kH: 0.945
New ak: 0.025


In [30]:
# 20 cm, frequency 1.07 Hz, Amplitude 0.15 V

H = 0.2
a = 0.0064
f = 1.07
w = 2 * np.pi * f
k = 2*np.pi/1.1
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 20 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 40 cm water depth---
H_new = 0.4
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 20 cm to 40 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 1.070 Hz
Original wave number: 5.712 rad/m
Original wavelength: 1.100 m
Original amplitude: 0.006 m
Original kH: 1.142
Original ak: 0.037
-----
Scaling results from 20 cm to 30 cm water depth:

New frequency: 0.874 Hz
New wave number: 3.808 rad/m
New wavelength: 1.650 m
New amplitude: 0.010 m
New kH: 1.142
New ak: 0.037
-----
Scaling results from 20 cm to 40 cm water depth:

New frequency: 0.757 Hz
New wave number: 2.856 rad/m
New wavelength: 2.200 m
New amplitude: 0.013 m
New kH: 1.142
New ak: 0.037


In [31]:
# 20 cm, frequency 1.22 Hz, Amplitude 0.15 V

H = 0.2
a = 0.0074
f = 1.22
w = 2 * np.pi * f
k = 2*np.pi/0.91
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 20 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 40 cm water depth---
H_new = 0.4
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 20 cm to 40 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 1.220 Hz
Original wave number: 6.905 rad/m
Original wavelength: 0.910 m
Original amplitude: 0.007 m
Original kH: 1.381
Original ak: 0.051
-----
Scaling results from 20 cm to 30 cm water depth:

New frequency: 0.996 Hz
New wave number: 4.603 rad/m
New wavelength: 1.365 m
New amplitude: 0.011 m
New kH: 1.381
New ak: 0.051
-----
Scaling results from 20 cm to 40 cm water depth:

New frequency: 0.863 Hz
New wave number: 3.452 rad/m
New wavelength: 1.820 m
New amplitude: 0.015 m
New kH: 1.381
New ak: 0.051


In [32]:
# 20 cm, frequency 1.37 Hz, Amplitude 0.15 V

H = 0.2
a = 0.0083
f = 1.37
w = 2 * np.pi * f
k = 2*np.pi/0.77
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 20 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 40 cm water depth---
H_new = 0.4
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 20 cm to 40 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 1.370 Hz
Original wave number: 8.160 rad/m
Original wavelength: 0.770 m
Original amplitude: 0.008 m
Original kH: 1.632
Original ak: 0.068
-----
Scaling results from 20 cm to 30 cm water depth:

New frequency: 1.119 Hz
New wave number: 5.440 rad/m
New wavelength: 1.155 m
New amplitude: 0.012 m
New kH: 1.632
New ak: 0.068
-----
Scaling results from 20 cm to 40 cm water depth:

New frequency: 0.969 Hz
New wave number: 4.080 rad/m
New wavelength: 1.540 m
New amplitude: 0.017 m
New kH: 1.632
New ak: 0.068


In [33]:
# 20 cm, frequency 1.53 Hz, Amplitude 0.15 V

H = 0.2
a = 0.009
f = 1.53
w = 2 * np.pi * f
k = 2*np.pi/0.63
kH = k*H
ak = a*k
lamb = 2 * np.pi / k

print()
print(f"Original frequency: {f:.3f} Hz")
print(f"Original wave number: {k:.3f} rad/m")
print(f"Original wavelength: {lamb:.3f} m")
print(f"Original amplitude: {a:.3f} m")
print(f"Original kH: {kH:.3f}")
print(f"Original ak: {ak:.3f}")
print("-----")

# ---Scaling to 30 cm water depth---
H_new = 0.3
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("Scaling results from 20 cm to 30 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")

# --- Scaling to 40 cm water depth---
H_new = 0.4
a_new = a*H_new/H
k_new = k*H/H_new
w_new = w * np.sqrt(H/H_new)
f_new = w_new/(2*np.pi)
kH_new = k_new*H_new
ak_new = a_new*k_new
lamb_new = 2 * np.pi / k_new

print("-----")
print("Scaling results from 20 cm to 40 cm water depth:")
print()
print(f"New frequency: {f_new:.3f} Hz")
print(f"New wave number: {k_new:.3f} rad/m")
print(f"New wavelength: {lamb_new:.3f} m")
print(f"New amplitude: {a_new:.3f} m")
print(f"New kH: {kH_new:.3f}")
print(f"New ak: {ak_new:.3f}")


Original frequency: 1.530 Hz
Original wave number: 9.973 rad/m
Original wavelength: 0.630 m
Original amplitude: 0.009 m
Original kH: 1.995
Original ak: 0.090
-----
Scaling results from 20 cm to 30 cm water depth:

New frequency: 1.249 Hz
New wave number: 6.649 rad/m
New wavelength: 0.945 m
New amplitude: 0.013 m
New kH: 1.995
New ak: 0.090
-----
Scaling results from 20 cm to 40 cm water depth:

New frequency: 1.082 Hz
New wave number: 4.987 rad/m
New wavelength: 1.260 m
New amplitude: 0.018 m
New kH: 1.995
New ak: 0.090


## Scaled waves

In [ ]:
# Scaled results from 40 cm depth to 30 cm and 20 cm depths
results_df_from_40cm

,f 40cm,a 40cm,k 40cm,λ 40cm,f 20cm,a 20cm,k 20cm,λ 20cm,f 30cm,a 30cm,k 30cm,λ 30cm,kH,ak
0,1.0,0.0180,4.303552,1.46,1.414214,0.0090,8.607103,0.730,1.154701,0.01350,5.738069,1.0950,1.721421,0.077464
1,0.9,0.0160,3.674377,1.71,1.272792,0.0080,7.348755,0.855,1.039230,0.01200,4.899170,1.2825,1.469751,0.058790
2,0.8,0.0140,3.095165,2.03,1.131371,0.0070,6.190330,1.015,0.923760,0.01050,4.126887,1.5225,1.238066,0.043332
3,0.7,0.0120,2.585673,2.43,0.989949,0.0060,5.171346,1.215,0.808290,0.00900,3.447564,1.8225,1.034269,0.031028
4,0.6,0.0100,2.129893,2.95,0.848528,0.0050,4.259787,1.475,0.692820,0.00750,2.839858,2.2125,0.851957,0.021299
5,0.5,0.0078,1.684500,3.73,0.707107,0.0039,3.369000,1.865,0.577350,0.00585,2.246000,2.7975,0.673800,0.013139


In [74]:
# Scaling from 30 cm to 20 and 40 cm

original_frequencies = [1.18, 1.06, 0.95, 0.83, 0.71, 0.59]
original_amplitudes  = [0.0182, 0.0161, 0.0145, 0.0122, 0.0102, 0.0084]
original_wavelengths = [1.06, 1.24, 1.46, 1.75, 2.17, 2.65]

original_wave_numbers = [2 * np.pi / l for l in original_wavelengths]

H_1 = 0.3  # original depth
H_2 = 0.2
H_3 = 0.4

results = []

for f, a, k, lamb in zip(original_frequencies, original_amplitudes, original_wave_numbers, original_wavelengths):
    w = 2 * np.pi * f

    # --- Scaling to 20 cm ---
    w_new_2 = w * np.sqrt(H_1 / H_2)
    f_new_2 = w_new_2 / (2 * np.pi)
    a_new_2 = a * H_2 / H_1
    k_new_2 = k * H_1 / H_2
    lamb_new_2 = 2 * np.pi / k_new_2
    kH_new_2 = k_new_2 * H_2
    ak_new_2 = a_new_2 * k_new_2

    # --- Scaling to 40 cm ---
    w_new_3 = w * np.sqrt(H_1 / H_3)
    f_new_3 = w_new_3 / (2 * np.pi)
    a_new_3 = a * H_3 / H_1
    k_new_3 = k * H_1 / H_3
    lamb_new_3 = 2 * np.pi / k_new_3
    kH_new_3 = k_new_3 * H_3
    ak_new_3 = a_new_3 * k_new_3

    results.append({
        'f 30 cm': f,
        'a 30 cm': a,
        'k 30 cm': k,
        'λ 30 cm': lamb,
        'f 20 cm': f_new_2,
        'a 20 cm': a_new_2,
        'k 20 cm': k_new_2,
        'λ 20 cm': lamb_new_2,
        'f 40 cm': f_new_3,
        'a 40 cm': a_new_3,
        'k 40 cm': k_new_3,
        'λ 40 cm': lamb_new_3,
        'kH': k * H_1,
        'ak': a * k,
    })

results_df_from_30cm = pd.DataFrame(results)

In [75]:
results_df_from_30cm

,f 30 cm,a 30 cm,k 30 cm,λ 30 cm,f 20 cm,a 20 cm,k 20 cm,λ 20 cm,f 40 cm,a 40 cm,k 40 cm,λ 40 cm,kH,ak
0,1.18,0.0182,5.927533,1.06,1.445199,0.012133,8.891300,0.706667,1.021910,0.024267,4.445650,1.413333,1.778260,0.107881
1,1.06,0.0161,5.067085,1.24,1.298230,0.010733,7.600627,0.826667,0.917987,0.021467,3.800314,1.653333,1.520125,0.081580
2,0.95,0.0145,4.303552,1.46,1.163508,0.009667,6.455327,0.973333,0.822724,0.019333,3.227664,1.946667,1.291065,0.062401
3,0.83,0.0122,3.590392,1.75,1.016538,0.008133,5.385587,1.166667,0.718801,0.016267,2.692794,2.333333,1.077117,0.043803
4,0.71,0.0102,2.895477,2.17,0.869569,0.006800,4.343216,1.446667,0.614878,0.013600,2.171608,2.893333,0.868643,0.029534
5,0.59,0.0084,2.371013,2.65,0.722599,0.005600,3.556520,1.766667,0.510955,0.011200,1.778260,3.533333,0.711304,0.019917
